# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library. It provides step-by-step examples for:

- Loading Croissant metadata and records
- Reviewing available record sets and fields by `@id`
- Extracting records into DataFrames
- Performing exploratory data analysis (EDA)
- Visualizing trends and relationships

### Dataset Source
This dataset is described using a Croissant schema, accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We'll reference all data entities such as record sets, fields, and columns by their `@id` for reproducibility and clarity.

In [ ]:
# Install `mlcroissant` if not already present
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset.
If you want to explore another dataset, simply change the schema URL below.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"TITLE: {dataset.metadata.name}")
print(f"DESCRIPTION: {dataset.metadata.description}")
print(f"CITATION: {getattr(dataset.metadata, 'cite_as', getattr(dataset.metadata, 'citeAs', None))}\n")
print("Published on:", getattr(dataset.metadata, 'datePublished', None))

## 2. Data Overview
Let's review available record sets and their fields. We use the record set and field `@id` identifiers as required for future steps.

The code below lists all record sets and, for each, prints its fields/columns by their `@id` and label.

In [ ]:
# Enumerate record sets and fields using their `@id`
record_sets = list(dataset.metadata.record_sets) if hasattr(dataset.metadata, 'record_sets') else []
if not record_sets:  # Try old-style attribute as fallback
    record_sets = getattr(dataset.metadata, 'recordSet', getattr(dataset.metadata, 'record_sets', []))

# If record_sets is a dict or has @id only, try to access actual objects
if record_sets and hasattr(record_sets[0], '@id'):
    # Already objects
    rs_objs = record_sets
elif record_sets and isinstance(record_sets[0], dict) and '@id' in record_sets[0]:
    rs_objs = record_sets
elif hasattr(dataset, 'record_set_objects'):
    rs_objs = dataset.record_set_objects.values()
else:
    rs_objs = []

record_set_ids = []

print("Available Record Sets:")
if not rs_objs:
    # Use Dataset interface to get all record set IDs
    for rs in dataset.metadata.to_json().get('recordSet', []):
        rs_id = rs.get('@id')
        print(f"- {rs_id}")
        record_set_ids.append(rs_id)
else:
    for rs in rs_objs:
        print(f"- @id: {rs['@id']} name: {rs.get('name', rs.get('label', ''))}")
        record_set_ids.append(rs['@id'])

# If record set details are not directly available, list IDs from JSON
if not record_set_ids:
    # fall back to schema extraction
    meta_json = dataset.metadata.to_json()
    for rs in meta_json.get('recordSet', []):
        rs_id = rs.get('@id')
        print(f"- {rs_id}")
        record_set_ids.append(rs_id)

# For each record set, print available fields/columns and their IDs
meta_json = dataset.metadata.to_json()
for rs in meta_json.get('recordSet', []):
    print(f"\nRecord Set @id: {rs.get('@id', None)}")
    # Fields are typically under 'field'
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', field) if isinstance(field, dict) else field
        field_label = field.get('label', field.get('name', '')) if isinstance(field, dict) else ''
        print(f"  - field @id: {field_id}, label: {field_label}")


## 3. Data Extraction
Now let's extract data for each record set into pandas DataFrames.

We'll use their `@id` identifiers as discussed above. If you're analyzing a specific field or record set, copy and use their `@id`.


In [ ]:
# List of record set IDs (update as found in the overview section). 
# For this dataset, let's extract all available record sets in the schema.
meta_json = dataset.metadata.to_json()
record_sets = [r.get('@id') for r in meta_json.get('recordSet', [])]

dataframes = {}

for record_set in record_sets:
    print(f"Loading records from record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        # If the records are not empty and are dicts
        if records and isinstance(records[0], dict):
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f" - Loaded {len(df)} records, columns: {df.columns.tolist()}")
        else:
            print(f" - No records found or records not tabular for {record_set}")
    except Exception as e:
        print(f" - Error loading records from {record_set}: {e}")

if dataframes:
    # Just show the first dataframe loaded
    first_rs = list(dataframes)[0]
    print(f"\nFirst DataFrame columns ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No tabular data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some exploratory analysis. We'll:
- Select a numeric field (e.g., molecular weight `MW` or peptide count)
- Filter for records above a threshold
- Normalize the numeric field
- Group by a categorical field if available

First, identify numeric and grouping fields from the DataFrame columns (all should be referenced by `@id`).

In [ ]:
# Replace these IDs with the actual IDs shown in the schema
# For demonstration, we'll try using field names like 'MW', 'peptide_count', or similar

first_record_set = list(dataframes)[0] if dataframes else None
if not first_record_set:
    raise RuntimeError("No dataframes loaded! Check previous cell output.")
df = dataframes[first_record_set]

# Try to guess a numeric field and a grouping field by common names
numeric_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['mw', 'weight', 'count', 'abundance', 'peptide', 'intensity', 'number'])]
print(f"Numeric field candidates: {numeric_field_candidates}")

# Use first numeric candidate
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Try a typical grouping field
group_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['group', 'sample', 'accession', 'type', 'category', 'label'])]
group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]
print(f"Grouping by: {group_field}")

# EDA steps
# Filtering records above a threshold
threshold = (df[numeric_field_id].dropna().mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10)

# Try casting numeric field if not already numeric
filtered_df = df.copy()
try:
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
except Exception:
    pass

filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field and get means
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, observed=True).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing mean of numeric fields):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields. For example, plot a histogram of the numeric field or a boxplot grouped by your group field.

Below we show how to plot a histogram and, if grouping is possible, a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not filtered_df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered, >{threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field in filtered_df.columns:
        # Only plot if not too many unique groups
        if filtered_df[group_field].nunique() < 30:
            plt.figure(figsize=(10,4))
            sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
            plt.xticks(rotation=45)
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.show()

## 6. Conclusion
This notebook reviewed the FAIR^2 mass spectrometry dataset, loaded its record sets (referenced by `@id`), and performed exploratory analyses on quantitative fields such as molecular weight or peptide counts. We visualized numeric distributions and, where relevant, group-wise summaries.

You can adjust fields and record sets by substituting different `@id` values as listed in the overview section, or build more advanced analysis and visualizations using the DataFrames above.

For more information, see the dataset's schema: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json